# Assignment 2 — What an embedding actually knows

**AI for Mobility (CGN 6933) ·individual work**

Three models, three training signals, one question asked three times: *what did this
space learn, and how would you know if it hadn't?*

| | Question | Pts |
|---|---|---|
| **Q1** | Explain and differentiate the terminology | 25 |
| **Q2** | Word2Vec magic — does it transfer to transportation? | 40 |
| **Q3** | Does CLIP understand transportation? | 35 |

**Everything is loaded for you.** You don't need to write the setup code. What you supply is the
*transportation content* — the word equations in Q2, the images and captions in Q3 —
and the interpretation. That is the part being graded.

**Provide your own images, and write your own intepretion without the help of AI**

---
Run **Runtime → Run all** once at the start. The first run downloads about 1 GB
(GloVe ~480 MB, CLIP ~600 MB) and takes a few minutes. Everything fits in one free
Colab session — no restarts needed.

In [9]:
NAME    = "Marvin Osei-Kuffour"   # <- your name
UNUMBER = "U52294607"   # <- your U-number
assert NAME and UNUMBER, "Fill in NAME and UNUMBER before running the rest."
print(f"Assignment 2 — {NAME} ({UNUMBER})")

Assignment 2 — Marvin Osei-Kuffour (U52294607)


## Setup — nothing here for you to change

Three cells. Run them in order and move on.

This notebook is **self-contained**: it installs everything it needs and runs the same
way on **Google Colab, macOS (Intel or Apple Silicon), and Windows**. You do not need
to install anything beforehand.

- **Easiest path — Google Colab.** Upload this notebook to
  [colab.research.google.com](https://colab.research.google.com), then Runtime → Run all.
  Nothing to install on your own machine.
- **On your own laptop.** Any Python 3.9+ with Jupyter. Cell 1 installs the rest.

**What it needs:** about **2 GB of free disk** (the models are cached after the first
run) and roughly **3 GB of free RAM**. No GPU — everything here runs on CPU.

In [10]:
# ============================================================================
#  CELL 1 — install everything. Safe to re-run; it skips what you already have.
# ============================================================================
import importlib, platform, re, subprocess, sys

# gensim>=4.4.0 is the one pin that really matters. gensim 4.3.x breaks on a current
# scientific stack in two different ways, and both errors are baffling if you meet
# them cold:
#     "cannot import name 'triu' from scipy.linalg"   -> SciPy removed it in 1.13
#     "numpy.dtype size changed, may indicate binary incompatibility"
#                                                     -> its C extensions predate NumPy 2
# Verified: gensim 4.3.3 raises the second one against numpy 2.x. Do not relax this pin.
REQUIRE = [
    ("numpy",        "numpy>=1.26",        (1, 26)),
    ("scipy",        "scipy>=1.11",        (1, 11)),
    ("gensim",       "gensim>=4.4.0",      (4, 4)),
    ("torch",        "torch>=2.0",         (2, 0)),
    ("torchvision",  "torchvision",        None),
    ("transformers", "transformers>=4.40", (4, 40)),
    ("PIL",          "pillow",             None),
    ("matplotlib",   "matplotlib",         None),
]


def _ver(mod):
    nums = re.findall(r"\d+", (getattr(mod, "__version__", "") or "").split("+")[0])
    return tuple(int(n) for n in nums[:2]) if nums else (0, 0)


def _pip(specs):
    # Use the %pip MAGIC when we are in a notebook: it installs into the kernel that is
    # actually running. Plain `!pip` can install into a DIFFERENT Python -- the classic
    # Windows / Anaconda failure where the install "succeeds" and the import still fails.
    try:
        ip = get_ipython()
    except NameError:
        ip = None
    if ip is not None:
        ip.run_line_magic("pip", "install -q " + " ".join(f'"{s}"' for s in specs))
    else:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *specs])


need = []
for _name, _spec, _min in REQUIRE:
    try:
        _m = importlib.import_module(_name)
        if _min and _ver(_m) < _min:
            need.append(_spec)
    except Exception:
        need.append(_spec)

print(f"Python {sys.version.split()[0]}  ·  {platform.system()} {platform.machine()}")
print(f"running in: {'Google Colab' if 'google.colab' in sys.modules else 'a local Jupyter kernel'}")
if need:
    print("\ninstalling: " + ", ".join(need))
    print("(first time only -- a few minutes)\n")
    _pip(need)
    print("\nInstall finished.")
    print("If CELL 2 below fails with an ImportError, restart the kernel")
    print("  (Colab: Runtime -> Restart session · Jupyter: Kernel -> Restart)")
    print("  and run these cells again. One restart is sometimes needed; never two.")
else:
    print("\nAll required packages are already present — nothing to install.")

Python 3.13.15  ·  Linux x86_64
running in: Google Colab

All required packages are already present — nothing to install.


In [11]:
# ============================================================================
#  CELL 2 — imports and helpers. If this fails, restart the kernel and re-run.
# ============================================================================
import os, sys, random, warnings
from pathlib import Path

# Windows: HuggingFace warns that it cannot make symlinks in its cache. Harmless here,
# and noisy, so silence it before transformers is imported.
os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")

import numpy as np
import matplotlib.pyplot as plt
import torch
from PIL import Image
warnings.filterwarnings("ignore")

random.seed(0); np.random.seed(0); torch.manual_seed(0)

GLOVE_ID = "glove-wiki-gigaword-300"       # 400,000 words x 300 dims
CLIP_ID  = "openai/clip-vit-base-patch32"  # 512-dim joint image-text space


def cos(a, b):
    a = np.asarray(a, dtype=float).ravel(); b = np.asarray(b, dtype=float).ravel()
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))


def clip_feats(out):
    # transformers 4.x returns a Tensor here; 5.x returns BaseModelOutputWithPooling.
    # Lab 4 calls this same helper `_as_tensor`.
    if torch.is_tensor(out):
        return out
    if hasattr(out, "pooler_output"):
        return out.pooler_output
    return out[0]


def l2(x):
    if torch.is_tensor(x):
        return x / x.norm(dim=-1, keepdim=True)
    return x / np.linalg.norm(x, axis=-1, keepdims=True)


import gensim
print(f"numpy {np.__version__} · torch {torch.__version__} · gensim {gensim.__version__}")
print("CPU is fine — nothing here needs a GPU.")

numpy 2.1.3 · torch 2.11.0+cpu · gensim 4.4.0
CPU is fine — nothing here needs a GPU.


In [12]:
# ============================================================================
#  CELL 3 — download and load the three models. First run only: ~1 GB.
#  Cached afterwards, so re-running this notebook later is fast.
# ============================================================================
import gensim.downloader as api
from transformers import CLIPModel, CLIPProcessor
import torchvision

print("loading GloVe word vectors ...")
glove = api.load(GLOVE_ID)

print("loading CLIP ...")
clip_model = CLIPModel.from_pretrained(CLIP_ID).eval()
clip_proc  = CLIPProcessor.from_pretrained(CLIP_ID)

print("loading ResNet-50 ...")
resnet = torchvision.models.resnet50(weights="IMAGENET1K_V2").eval()

print()
print(f"  GloVe   {len(glove.index_to_key):,} words x {glove.vector_size} dims")
print(f"  CLIP    {sum(p.numel() for p in clip_model.parameters()):,} parameters")
print(f"  ResNet  {sum(p.numel() for p in resnet.parameters()):,} parameters")
print("\nAll three loaded. Nothing else to install.")

loading GloVe word vectors ...
loading CLIP ...


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

loading ResNet-50 ...

  GloVe   400,000 words x 300 dims
  CLIP    151,277,313 parameters
  ResNet  25,557,032 parameters

All three loaded. Nothing else to install.


---
# Q1 — Explain and differentiate the following terminologies  (25 pts)

Eight terms get used for roughly the same row of numbers, and papers switch between
them without warning. You cannot read a methods section until you know which
distinctions are real and which are one research community's habit.

**Run the cell below first.** It prints what CLIP and ResNet-50 are actually made of,
and the shape of the tensor at every stage. Fill the table *against that output* — a
definition with no tensor attached is the kind of answer any chatbot gives, and it
scores accordingly.

In [13]:
# ---- GIVEN: the anatomy of both models, for you to read off ------------------
print("=" * 74)
print("ResNet-50 — top-level modules, and the tensor after each one")
print("=" * 74)
x = torch.randn(1, 3, 224, 224)
with torch.no_grad():
    h = x
    for name, mod in resnet.named_children():
        n = sum(p.numel() for p in mod.parameters())
        if name == "fc":
            h = torch.flatten(h, 1)
        h = mod(h)
        print(f"  {name:10s} {mod.__class__.__name__:18s} {n:>11,} params"
              f"   -> {tuple(h.shape)}")

print()
print("=" * 74)
print("CLIP — two towers landing in ONE space")
print("=" * 74)
with torch.no_grad():
    t_out = clip_model.get_text_features(
        **clip_proc(text=["a wet road", "a dry road"], return_tensors="pt", padding=True))
    i_out = clip_model.get_image_features(
        **clip_proc(images=Image.fromarray(
            (np.random.rand(224, 224, 3) * 255).astype("uint8")), return_tensors="pt"))
for name, mod in clip_model.named_children():
    n = sum(p.numel() for p in mod.parameters())
    print(f"  {name:20s} {mod.__class__.__name__:24s} {n:>11,} params")
print()
print(f"  text  features -> {tuple(clip_feats(t_out).shape)}")
print(f"  image features -> {tuple(clip_feats(i_out).shape)}")
print()
print("  Both towers end at the SAME width. That is what 'joint space' means, and it")
print("  is why a cosine between an image and a sentence is defined at all.")

ResNet-50 — top-level modules, and the tensor after each one
  conv1      Conv2d                   9,408 params   -> (1, 64, 112, 112)
  bn1        BatchNorm2d                128 params   -> (1, 64, 112, 112)
  relu       ReLU                         0 params   -> (1, 64, 112, 112)
  maxpool    MaxPool2d                    0 params   -> (1, 64, 56, 56)
  layer1     Sequential             215,808 params   -> (1, 256, 56, 56)
  layer2     Sequential           1,219,584 params   -> (1, 512, 28, 28)
  layer3     Sequential           7,098,368 params   -> (1, 1024, 14, 14)
  layer4     Sequential          14,964,736 params   -> (1, 2048, 7, 7)
  avgpool    AdaptiveAvgPool2d            0 params   -> (1, 2048, 1, 1)
  fc         Linear               2,049,000 params   -> (1, 1000)

CLIP — two towers landing in ONE space
  text_model           CLIPTextModel             63,165,952 params
  vision_model         CLIPVisionModel           87,456,000 params
  visual_projection    Linear            

## Your table

**Double-click this cell to edit it.**


| # | Term | What it means, in your own words | If exists, where it is in the model above | The distinction that matters |
|---|---|---|---|---|
| 1 | **Encoder — decoder vs backbone-head** | | | |
| 3 | **Encoder vs. backbone** | | | |
| 4 | **Representation / latents** | | | |
| 5 | **Representation vs. features** | | | |
| 6 | **Embedding space** | | | |
| 7 | **Hidden layer vs. latent layer** | | | |
| 8 | **Representation vs. embedding vs. latent output** | | | |



---
# Q2 — Word2Vec magic  (40 pts)

Word2Vec is said to capture embeddings that carry *semantics*. The usual demonstration
is that a relationship becomes a **direction** you can add and subtract:

```
vec("king") − vec("man") + vec("woman")   ≈   vec("queen")
```

Note the operation: you **subtract** to isolate a relationship, then **add** it
somewhere else. (`king + man` is a different thing entirely, and it returns a
plausible-looking answer — the first trap below.)

**The question:** does that transfer to transportation? We have our own famous identity,
the one everyone in this room can recite:

> **q = k · v**  —  flow = density × speed

Does *it* hold in the embedding?

## Q2.1 · The warm-up, given  

In [14]:
# ---- GIVEN -------------------------------------------------------------------
def vec(w):
    if w.lower() not in glove.key_to_index:
        raise KeyError(f"{w!r} is not in this vocabulary — that is a finding, see Q2.4")
    return glove[w.lower()]


def top(positive, negative=(), n=3):
    return glove.most_similar(positive=[w.lower() for w in positive],
                              negative=[w.lower() for w in negative], topn=n)


print("the analogy, done correctly (a DIFFERENCE):")
print("  king - man + woman ->",
      [(w, round(s, 3)) for w, s in top(["king", "woman"], ["man"])])
print(f"  cos(king - man + woman, queen) = "
      f"{cos(vec('king') - vec('man') + vec('woman'), vec('queen')):.4f}")

print("\nthe same words, added instead of subtracted:")
print("  king + man    ->", [(w, round(s, 3)) for w, s in top(["king", "man"])])
print("  queen + woman ->", [(w, round(s, 3)) for w, s in top(["queen", "woman"])])
print(f"  cos(king + man, queen + woman) = "
      f"{cos(vec('king') + vec('man'), vec('queen') + vec('woman')):.4f}"
      f"   <-- looks like a result")

the analogy, done correctly (a DIFFERENCE):
  king - man + woman -> [('queen', 0.671), ('princess', 0.543), ('throne', 0.539)]
  cos(king - man + woman, queen) = 0.6896

the same words, added instead of subtracted:
  king + man    -> [('brother', 0.617), ('father', 0.615), ('son', 0.594)]
  queen + woman -> [('mother', 0.694), ('her', 0.639), ('girl', 0.638)]
  cos(king + man, queen + woman) = 0.6530   <-- looks like a result


**Q2.1 answer.** That last cosine is not small. Explain in one or two sentences why it
is nonetheless evidence of nothing.

> *Your answer:*

## Q2.2 · Design your own transportation equations

Design **three** operations of your own that make sense in transportation. At least one
must **cross modes** (road → rail, road → air, road → water).

Two shapes are available:

```
relational      taxiway  ~  highway - car + runway     a relationship moved to another mode
```

**Predict the answer before running each one.**

In [15]:
# --- YOUR TURN ---------------------------------------------------------------
# One line per equation:
#   (target, [words to add], [words to subtract], "what you expect and why")
#
# MY_EQUATIONS = [
#     ("taxiway", ["highway", "runway"], ["car"], "road-to-air facility transfer"),
#     ("trains",  ["bus", "train"],      ["road"], "transit vehicle by guideway"),
#     ("delay",   ["congestion", "queue"], [],     "compositional: delay from both"),
# ]

MY_EQUATIONS = []

assert len(MY_EQUATIONS) >= 3, "Design at least three equations."
for target, add, sub, note in MY_EQUATIONS:
    test_equation(target, add, sub, note=note)

AssertionError: Design at least three equations.

**Q2.3 write-up.** For each equation: did it land where you predicted?

**Report your misses.** Three hits and no misses means you stopped searching as soon as
the model flattered you, and it will be graded that way. For each miss, say whether the
model is missing the *relationship* or missing the *word* — different failures, different
fixes.

> *Your answer:*

## Q2.3 · Why it behaved that way  

In [ ]:
# ---- GIVEN: what do these words mean TO THIS MODEL? -------------------------
TERMS = ["density", "headway", "platoon", "arterial", "occupancy", "saturation",
         "shockwave", "congestion", "queue", "bottleneck", "signalized", "los", "veh"]
for t in TERMS:
    if t in glove.key_to_index:
        print(f"  {t:12s} {[w for w, _ in glove.most_similar(t, topn=5)]}")
    else:
        print(f"  {t:12s} *** not in vocabulary ***")

print("\n  and some terms you might expect to be there:")
for t in ["aadt", "vph", "free_flow", "jam_density", "level_of_service", "stop_sign"]:
    print(f"  {t:20s} {'in vocab' if t in glove.key_to_index else 'KeyError'}")

**Q2.3 answer.** These do **not** all behave the same way, and the split is the point.
Sort them into three groups and name the sense the model learned:

| Group | Terms | The sense it actually has |
|---|---|---|
| Has *our* transportation sense | | |
| Has a *different field's* sense | | |
| Has no useful sense at all | | |

Then: several terms in the second list raise `KeyError`. **Why that particular set?**
The reason is a property of how this model's vocabulary was built, and it is one line.
One of them *is* in the vocabulary — what does that one tell you?

> *Your answer:*

## Q2.4 · The write-up  

In [ ]:
# ---- GIVEN: two facts that have to be explained together --------------------
print("Fact 1 — adding two number words:")
print("  two + three ->", [w for w, _ in top(["two", "three"], n=5)])
print(f"  cos(two + three, five) = {cos(vec('two') + vec('three'), vec('five')):.4f}")

print("\nFact 2 — three physical identities, ALL TRUE, same additive test:")
for tgt, a, b, form in [("distance", "speed", "time", "d = v * t"),
                        ("power", "voltage", "current", "P = V * I"),
                        ("area", "length", "width", "A = l * w")]:
    print(f"  cos({tgt:9s}, {a} + {b:8s}) = {cos(vec(tgt), vec(a) + vec(b)):.4f}    {form}")

**Q2.5 — 250 words.** Fact 1 looks exactly like the model doing arithmetic. Fact 2 shows
three identities that are *all true* scoring very differently from one another.

Explain both. Then state the general rule for what this geometry does and does not
encode, and name **one** transportation task where that rule makes a general-purpose
word embedding the wrong tool — and say what you would use instead.

> *Your answer:*

---
# Q3 — Does CLIP understand transportation?  (35 pts)

> **CLIP does not write captions.** It is two encoders trained with a contrastive
> objective — an image tower and a text tower landing in one shared 512-dimensional
> space. There is **no text decoder** anywhere in it. CLIP can only *score* how well a
> given piece of text matches a given image.

So you will not ask CLIP what it sees. You **write the captions yourself**, from a plain
general description up to one only a transportation engineer would write, and measure
which one CLIP places closest to the image. **Where the ladder stops climbing is where
CLIP's understanding stops.**

The model is already loaded. For the encode-and-score pattern in its original form, see
[Lab 4 — CLIP vs. Traditional Computer Vision](https://ai4mobility.github.io/module1/lab4_clip_vs_traditional_cv.html);
`clip_feats` here is Lab 4's `_as_tensor`.

## Q3.1 · Two images of your own  

Provide two images of youru own. They can be dashcam frame, FL511 screenshot, or a phone photo. They must differ in one way, and
**this is the experimental design**:

| | What it has to be |
|---|---|
| **obvious** | the transportation situation *fills the frame* — a flooded roadway, a crash scene, an unmissable work zone, gridlock |
| **subtle** | the safety-relevant thing is *a small part* of an otherwise ordinary picture — one pedestrian near a curb, a scooter in the far lane, a vehicle on a shoulder |

Two images of the same kind give you one answer twice, and you will conclude the wrong
thing from it.

**Getting your images in.** On Colab, run the upload cell below and pick both files.
On your own laptop, put them in the same folder as this notebook (or set `IMAGE_DIR`
to wherever they are — a plain Windows path like `C:\Users\you\Pictures` is fine).

In [18]:
# ---- GIVEN: Colab upload helper. On a laptop this cell does nothing. ---------
if "google.colab" in sys.modules:
    from google.colab import files
    print("Pick BOTH of your images in the dialog below.")
    files.upload()          # lands them next to the notebook
else:
    print("Not on Colab — just put your two images in the folder set as IMAGE_DIR.")

Pick BOTH of your images in the dialog below.


Saving 16x9_M-34.jpg to 16x9_M-34.jpg
Saving scooter.jpg to scooter.jpg


In [17]:
# --- YOUR TURN ---------------------------------------------------------------
IMAGE_DIR = "/content/"     # "." = same folder as this notebook. A full path also works:
                    #   Windows  r"C:\Users\you\Pictures"
                    #   macOS    "/Users/you/Pictures"

MY_IMAGES = (["/content/16x9_M-34.jpg","obvious", "where / when / lighting", "the cue that matters"),("/content/scooter.jpg","subtle",  "where / when / lighting", "the cue that matters")

]

# ---- from here down it is given ---------------------------------------------
assert len(MY_IMAGES) == 2 and {r[1] for r in MY_IMAGES} == {"obvious", "subtle"}, \
    "Exactly two images: one tagged 'obvious', one tagged 'subtle'."

folder = Path(IMAGE_DIR).expanduser()


def _find(fn):
    # Tolerate case differences and a missing/altered extension -- Windows and macOS
    # disagree about case, and phones rename things .JPG / .jpeg / .HEIC.
    p = folder / fn
    if p.exists():
        return p
    stem = Path(fn).stem.lower()
    for cand in sorted(folder.iterdir()):
        if cand.is_file() and cand.stem.lower() == stem \
                and cand.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".webp"}:
            return cand
    here = [c.name for c in sorted(folder.iterdir())
            if c.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".webp"}]
    raise FileNotFoundError(
        f"Could not find {fn!r} in {folder.resolve()}\n"
        f"Image files actually in that folder: {here if here else '(none)'}\n"
        f"Fix the filename in MY_IMAGES, or point IMAGE_DIR at the right folder.")


IMG = {}
for fn, kind, meta, cue in MY_IMAGES:
    path = _find(fn)
    IMG[kind] = (Image.open(path).convert("RGB"), path.name)
    print(f"  {kind:8s} {path.name:26s} {meta}")
    print(f"  {'':8s} {'':26s} cue: {cue}")

fig, axes = plt.subplots(1, 2, figsize=(9, 3.6))
for ax, k in zip(axes, ["obvious", "subtle"]):
    ax.imshow(IMG[k][0]); ax.set_title(f"{k} — {IMG[k][1]}", fontsize=10); ax.axis("off")
plt.tight_layout(); plt.show()

AssertionError: Exactly two images: one tagged 'obvious', one tagged 'subtle'.

## Q3.2 · Your description, written blind  

**Before running any scoring**, write two or three sentences per image: what a traffic
engineer would note — what is happening, what the risk is, what you would want a system
to flag. Do not edit it afterwards. It is only a baseline if it was written blind.

**obvious —**
> *Your description:*

**subtle —**
> *Your description:*

## Q3.3 · Design and write a caption ladder  

For each image write **at least five captions**, ordered from a general description up
to a transportation-specific one.

Then one more: a caption that is **deliberately wrong about the one safety-relevant
fact** — same length and register as your most specific caption, but wrong.
*"on the sidewalk"* where the truth is *"in the travel lane."*

That last caption is the whole measurement. If CLIP scores it the same as the true one,
CLIP never saw the distinction that decides whether the frame is a near-miss or a
non-event.

> **Read the margin, not the score.** CLIP's raw image–text cosines sit in a narrow
> band — almost everything lands around 0.15–0.30, including text with nothing to do
> with the picture. An absolute score of 0.28 means nothing on its own. Only differences
> *between* captions on the *same* image carry information.

In [ ]:
# ---- GIVEN: scoring, ranking, and every control, computed for you ------------
@torch.no_grad()
def clip_score(image, texts):
    ti = clip_proc(text=list(texts), return_tensors="pt", padding=True)
    ii = clip_proc(images=image, return_tensors="pt")
    t = l2(clip_feats(clip_model.get_text_features(**ti)))
    i = l2(clip_feats(clip_model.get_image_features(**ii)))
    return (i @ t.T).squeeze(0).numpy()


def run_ladder(kind, other_kind):
    captions, wrong = LADDERS[kind]
    image, fn = IMG[kind]
    texts = list(captions) + [wrong]
    s = clip_score(image, texts)
    lad_s = s[:-1]
    lad_w = np.array([len(c.split()) for c in captions])

    print("=" * 78)
    print(f"  {kind.upper()} — {fn}")
    print("=" * 78)
    print(f"  {'#':>3s} {'score':>7s} {'words':>6s}  caption")
    for r in np.argsort(-s):
        tag = "   <-- DELIBERATELY WRONG" if r == len(texts) - 1 else ""
        print(f"  {r+1:>3d} {s[r]:7.4f} {len(texts[r].split()):6d}  {texts[r][:42]}{tag}")

    print(f"\n  ladder winner              : caption {int(np.argmax(lad_s)) + 1}"
          f"  ({lad_s.max():.4f})")
    print(f"  climbs the whole way up?   : {bool(np.all(np.diff(lad_s) > 0))}")
    print(f"  corr(score, word count)    : {np.corrcoef(lad_s, lad_w)[0, 1]:+.3f}")
    print(f"\n  best TRUE caption          : {lad_s.max():.4f}")
    print(f"  DELIBERATELY WRONG caption : {s[-1]:.4f}")
    print(f"  >>> true-minus-wrong MARGIN = {lad_s.max() - s[-1]:+.4f} <<<")

    # automatic control: score THIS image against the OTHER image's captions
    other_caps = LADDERS[other_kind][0]
    if other_caps:
        so = clip_score(image, list(other_caps))
        print("\n  CROSS-IMAGE CONTROL — the other image's captions, scored on this one")
        print(f"     best foreign caption   : {so.max():.4f}")
        print(f"     your best true caption : {lad_s.max():.4f}"
              f"   -> beats foreign by {lad_s.max() - so.max():+.4f}")
        print("     (a small gap means your captions are not describing THIS picture)")
    print()
    return s


print("scoring helper ready")

In [ ]:
# --- YOUR TURN ---------------------------------------------------------------
# Captions ordered general -> transportation-specific, then ONE deliberately wrong one.
#
# LADDERS = {
#     "obvious": ([
#         "a photo",
#         "a street with water on it",
#         "a flooded street in a commercial area after a storm",
#         "cars partly submerged in floodwater",
#         "an impassable flooded roadway with submerged vehicles blocking the travel lanes",
#     ], "vehicles driving normally through light rain on a clear roadway"),
#
#     "subtle": ([...], "..."),
# }

LADDERS = {
    "obvious": ([], ""),
    "subtle":  ([], ""),
}

for k in ["obvious", "subtle"]:
    caps, wrong = LADDERS[k]
    assert len(caps) >= 5 and wrong, f"{k}: need 5+ captions and one deliberately wrong one."

for k, other in [("obvious", "subtle"), ("subtle", "obvious")]:
    run_ladder(k, other)

**Q3.3 answer.** For each image:

- Which caption won, and **does the score climb all the way up the ladder or stop
  somewhere?** Say where it stops and why you think it stops there.
- What is the **true-minus-wrong margin**, and where did the wrong caption rank?
- What does the word-count correlation suggest — and does the cross-image control
  support that reading or undercut it?

> *Your answer:*

## Q3.4 · Your verdict

**250 words.** Put your Q3.2 descriptions next to the numbers.

- **Did your two images behave the same way?** Compare the true-minus-wrong margin on
  the obvious image against the subtle one. If they differ, say what that tells you
  about when zero-shot CLIP is worth using — and which of the two cases a
  safety-alerting system actually needs.
- Name **two specific things** you wrote in Q3.2 that no caption could get CLIP to score
  reliably, and say what you would build to detect each.
- Plainly: **is CLIP capturing transportation-domain specifics such as safety cues, or
  is it producing a good general scene description that happens to contain
  transportation words?** Defend the answer with a margin, not an impression.

> *Your answer:*

---
## AI-use disclosure (required)

Use AI freely for code — none of the plumbing here is what is being assessed. The Q1
table is the exception: it is graded on the tensor names and shapes from **your** run,
and a generic answer with no shapes in it scores zero however well written.

**A warning specific to this assignment:** ask a chatbot whether `flow = density × speed`
holds in word2vec and it will give you a confident, plausible, entirely fabricated
answer — in either direction, depending on how you phrase the question. Q2 is settled by
*your* numbers and *your* baselines. Anything you cannot point at an output cell for is
not a result.

In [ ]:
AI_USE = '''
What I used AI for:


One place it was wrong or unhelpful:


For Q1, what I wrote myself and what I changed after checking:

'''
assert len(AI_USE.strip()) > 120, "Fill in the disclosure block."
print(AI_USE)